# Bigram / Mini Transformer Language Model

Character-level language model trained on Tiny Shakespeare, built from a bigram baseline up to a small Transformer.

**Improvements vs original:** larger context & model, GELU MLP, weight tying, higher LR + warmup, longer training, temperature/top-k sampling for cleaner generation.

## 1. Setup

In [ ]:
import math
import os

import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu
Torch 2.13.0+cpu | threads=4


## 2. Config

In [ ]:
block_size = 64        # was 32 — longer context
n_embd = 128           # was 64
n_head = 4             # same head count, larger head size
n_layer = 6            # was 4
dropout = 0.2          # was 0.1

batch_size = 32
learning_rate = 1e-3   # was 3e-4
max_iters = 5000
warmup_iters = 200
eval_interval = 250
eval_iters = 40
grad_clip = 1.0

temperature = 0.8
top_k = 40

checkpoint_path = "model.pt"

BASELINE_TEST_LOSS = 1.9125
BASELINE_PARAMS_K = 209.7

## 3. Load Data

In [ ]:
with open("tiny shakespeare.txt", "r") as f:
    text = f.read()

## 4. Build the Character Vocabulary

In [ ]:
vocab = sorted(list(set(text)))
vocab_size = len(vocab)

## 5. Custom Tokenizer

In [2]:
encod = {ch: i for i, ch in enumerate(vocab)}
decod = {i: ch for i, ch in enumerate(vocab)}

encode = lambda s: [encod[i] for i in s]
decode = lambda l: ''.join([decod[i] for i in l])

print(encode("hello world"))
decode(encode("hello world"))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


## 6. Train / Test Split

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}")
if data.ndim > 1:
    data = data.flatten()
    print(f"Flattened data shape: {data.shape}")

n = int(0.9 * len(data))
train_set = data[:n]
test_set = data[n:]

Data shape: torch.Size([1115394])
10.000053792650847


## 7. Batching

Samples a fresh random batch on every call instead of pre-materializing every window in memory.

In [4]:
def get_batch(split):
    source = train_set if split == "train" else test_set
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print(f"xb shape: {xb.shape}")
print(f"yb shape: {yb.shape}")

xb shape: torch.Size([32, 64])
yb shape: torch.Size([32, 64])


## 8. Transformer Building Blocks

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


def top_k_logits(logits, k):
    if k is None or k <= 0:
        return logits
    v, _ = torch.topk(logits, min(k, logits.size(-1)))
    return logits.masked_fill(logits < v[:, [-1]], float("-inf"))

## 9. Baseline: Bigram Language Model

In [5]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for i in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

bigram_model = BigramLanguageModel(vocab_size)
xb, yb = get_batch("train")
logits, loss = bigram_model(xb, yb)
print(f"Bigram Logits shape: {logits.shape}")
print(f"Bigram Loss: {loss}")

Bigram Logits shape: torch.Size([2048, 65])
Bigram Loss: 4.67


## The Transformer

In [6]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying: share token embeddings with output projection
        self.lm_head.weight = self.token_embedding_table.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            logits = top_k_logits(logits, top_k)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        self.train()
        return idx


model = LanguageModel(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)
xb, yb = get_batch("train")
logits, loss = model(xb, yb)
n_params = sum(p.numel() for p in model.parameters())
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss}")
print(f"Params: {n_params / 1e3:.1f}K (baseline was {BASELINE_PARAMS_K}K)")

Logits shape: torch.Size([32, 64, 65])
Loss: 4.19
Params: 1204.1K (baseline was 209.7K)


## 11. Prediction Before Training

In [7]:
start_context = torch.zeros((1, 1), dtype=torch.long, device=device)

print("--- BEFORE TRAINING ---")
untrained_output = model.generate(
    start_context.clone(), max_new_tokens=300, temperature=1.0, top_k=None
)[0].tolist()
print(decode(untrained_output))

--- BEFORE TRAINING ---
:Zy3k&MHgo,?w; hwNCTKqo?hu g jHJafSNTgTof TRqdV


## 12. Training Loop

Loss is averaged over `eval_iters` batches for a stable estimate. Gradient clipping and a cosine learning-rate schedule are applied every step.

In [8]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "test"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def get_lr(it):
    # Linear warmup then cosine decay to 10% of peak LR
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    progress = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    return learning_rate * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.99), weight_decay=0.1)

train_losses = []
test_losses = []
best_test = float("inf")

for i in range(max_iters):
    lr = get_lr(i)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    if i % eval_interval == 0 or i == max_iters - 1:
        losses = estimate_loss()
        train_losses.append(losses["train"])
        test_losses.append(losses["test"])
        print(
            f"Iteration {i}: Train Loss {losses['train']:.4f} | "
            f"Test Loss {losses['test']:.4f} | LR {lr:.2e}"
        )
        if losses["test"] < best_test:
            best_test = losses["test"]
            torch.save(
                {
                    "model": model.state_dict(),
                    "config": {
                        "vocab_size": vocab_size,
                        "n_embd": n_embd,
                        "n_head": n_head,
                        "n_layer": n_layer,
                        "block_size": block_size,
                        "dropout": dropout,
                    },
                    "test_loss": best_test,
                },
                checkpoint_path,
            )

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

print(f"Final step loss: {loss.item():.4f}")
print(f"Best test loss: {best_test:.4f}  (baseline was {BASELINE_TEST_LOSS:.4f})")
print(f"Improvement: {BASELINE_TEST_LOSS - best_test:.4f} lower test loss")

Iteration 0: Train 4.1922 | Test 4.1904 | LR 5.00e-06 | 0.1m
Iteration 250: Train 2.4206 | Test 2.4253 | LR 1.00e-03 | 1.5m
Iteration 500: Train 2.1760 | Test 2.2041 | LR 9.91e-04 | 3.0m
Iteration 750: Train 2.0122 | Test 2.0769 | LR 9.71e-04 | 4.5m
Iteration 1000: Train 1.8980 | Test 1.9914 | LR 9.40e-04 | 6.0m
Iteration 1250: Train 1.8219 | Test 1.9375 | LR 8.98e-04 | 7.7m
Iteration 1500: Train 1.7745 | Test 1.9040 | LR 8.47e-04 | 9.2m
Iteration 1750: Train 1.7058 | Test 1.8522 | LR 7.88e-04 | 10.7m
Iteration 2000: Train 1.6790 | Test 1.8388 | LR 7.22e-04 | 12.3m
Iteration 2250: Train 1.6492 | Test 1.8021 | LR 6.52e-04 | 13.8m
Iteration 2500: Train 1.6069 | Test 1.7753 | LR 5.79e-04 | 17.1m
Iteration 2750: Train 1.5907 | Test 1.7554 | LR 5.06e-04 | 26.8m
Iteration 3000: Train 1.5561 | Test 1.7490 | LR 4.34e-04 | 28.2m
Iteration 3250: Train 1.5385 | Test 1.7257 | LR 3.64e-04 | 29.6m
Iteration 3500: Train 1.5264 | Test 1.7014 | LR 3.00e-04 | 31.1m
Iteration 3750: Train 1.5012 | Test 1.

## 13. Plot Loss Curve

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="train_loss")
plt.plot(test_losses, label="test_loss")
plt.axhline(BASELINE_TEST_LOSS, color="gray", linestyle="--", label=f"baseline test ({BASELINE_TEST_LOSS})")
plt.xlabel(f"Evaluation step (every {eval_interval} iterations)")
plt.ylabel("Loss")
plt.title("Training Progress")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=120)
plt.show()
print("Saved loss_curve.png")

Saved loss_curve.png


## 14. Save & Reload Checkpoint

In [10]:
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
cfg = ckpt["config"]
loaded_model = LanguageModel(
    cfg["vocab_size"], cfg["n_embd"], cfg["n_head"], cfg["n_layer"], cfg["block_size"], cfg["dropout"]
).to(device)
loaded_model.load_state_dict(ckpt["model"])
loaded_model.eval()
print(f"Loaded best checkpoint (test loss={ckpt['test_loss']:.4f})")

Loaded best checkpoint (test loss=1.6417)


## 15. Prediction After Training

In [11]:
print("--- AFTER TRAINING (temperature/top-k sampling) ---")
torch.manual_seed(42)
trained_output = loaded_model.generate(
    start_context.clone(),
    max_new_tokens=1000,
    temperature=temperature,
    top_k=top_k,
)[0].tolist()
print(decode(trained_output))

print("\n=== COMPARISON ===")
print(f"Baseline test loss : {BASELINE_TEST_LOSS:.4f}")
print(f"Improved test loss : {ckpt['test_loss']:.4f}")
print(f"Delta              : {BASELINE_TEST_LOSS - ckpt['test_loss']:+.4f}")
print(f"Baseline params    : {BASELINE_PARAMS_K}K")
print(f"Improved params    : {n_params / 1e3:.1f}K")
print(f"Context length     : 32 -> {block_size}")
print(f"Embedding / layers : 64x4 -> {n_embd}x{n_layer}")

--- AFTER TRAINING (temperature/top-k sampling) ---

KING RICHARD III:
God you we not only noble counter eyes are desire,
To dreamny we made the no heart, that is u
cersit the prection, he we like to the bound-choose?

MENENIUS:
So thus not cheek and do me be died him.

First Citizen:
I was came, to no before and were country be;
And by the many something do the passed
To enought. Why for I am not stribunes by away!

ROMEO:
Now, if I saw be that is said, he have a warrd's bout
Write live, the boys and talk'd that swears smoonguest

LADY ANNE:
We will I know!
Shall dange and that, I this chae is call it him

FRIAR LAURENCE:
What this king dleseng of; and neeXed well,
How master at I for thy will for murdeers from our percies;

=== COMPARISON ===
Baseline test loss : 1.9125
Improved test loss : 1.6417
Delta              : +0.2708
Baseline params    : 209.7K
Improved params    : 1204.1K
Context length     : 32 -> 64
Embedding / layers : 64x4 -> 128x6


## 16. Before vs After

| Metric | Baseline (original notebook) | Improved |
|--------|------------------------------|----------|
| Test loss | **1.9125** | **1.6417** (−0.27) |
| Params | 209.7K | 1204.1K |
| Context (`block_size`) | 32 | 64 |
| Width × depth | 64 embd × 4 layers | 128 embd × 6 layers |
| Sample quality | Broken words, weak names | Recognizable speakers (ROMEO, MENENIUS, …) and play-like structure |

**What changed to improve accuracy/output**
1. Larger model + longer context  
2. GELU MLP (was ReLU)  
3. Weight tying (embed ↔ lm_head) + better init  
4. LR warmup + cosine decay, AdamW weight decay  
5. Best-checkpoint saving (not just last step)  
6. Temperature 0.8 + top-k 40 sampling  

Re-run training with the project venv:

```powershell
.\.env\Scripts\python.exe run_train.py
```
